In [ ]:
import os
os.environ["GROQ_API_KEY"] = "add your key here"

## Install libraries

In [4]:
!pip install -q youtube-transcript-api langchain-community langchain-groq \
               langchain-huggingface sentence-transformers faiss-cpu python-dotenv langchain

In [7]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from youtube_transcript_api.proxies import WebshareProxyConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


/tmp/ipykernel_15341/2406505708.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "BmYv8XGl-YU" # only the ID, not full URL
try:

    ytt_api = YouTubeTranscriptApi(
        proxy_config=WebshareProxyConfig(
            proxy_username="your-user",
            proxy_password="your-pass"))
    fetched = ytt_api.fetch(video_id)

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in fetched)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [32]:
# Upload transcript.txt first using files.upload()

with open("transcript.txt", "r", encoding="utf-8") as f:
    transcript = f.read()

print("Transcript loaded successfully!")
print("Characters:", len(transcript))
print("\nPreview:\n")
print(transcript[:1000])

Transcript loaded successfully!
Characters: 21821

Preview:

00:31 President Faust, Board of Overseers, faculty,
alumni, friends, proud parents, members of
00:40 the ad board, and graduates of the greatest
university in the world,
00:55 I'm honored to be with you today because,
let's face it, you accomplished something
01:01 I never could.
01:04 If I get through this speech, it'll be the
first time I actually finish something at
01:10 Harvard.
01:11 Class of 2017, congratulations!
01:17 I'm an unlikely speaker, not just because
I dropped out, but because we're technically
01:27 in the same generation.
01:29 We walked this yard less than a decade apart,
studied the same ideas and slept through the
01:35 same Ec10 lectures.
01:38 We may have taken different paths to get here,
especially if you came all the way from the
01:43 Quad, but today I want to share what I've
learned about our generation and the world
01:50 we're building together.
01:52 But first, the last couple of days have bro

## Step 1b - Indexing (Text Splitting)

In [33]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [34]:
len(chunks)

27

In [35]:
chunks[0], chunks[1]

(Document(metadata={}, page_content="00:31 President Faust, Board of Overseers, faculty,\nalumni, friends, proud parents, members of\n00:40 the ad board, and graduates of the greatest\nuniversity in the world,\n00:55 I'm honored to be with you today because,\nlet's face it, you accomplished something\n01:01 I never could.\n01:04 If I get through this speech, it'll be the\nfirst time I actually finish something at\n01:10 Harvard.\n01:11 Class of 2017, congratulations!\n01:17 I'm an unlikely speaker, not just because\nI dropped out, but because we're technically\n01:27 in the same generation.\n01:29 We walked this yard less than a decade apart,\nstudied the same ideas and slept through the\n01:35 same Ec10 lectures.\n01:38 We may have taken different paths to get here,\nespecially if you came all the way from the\n01:43 Quad, but today I want to share what I've\nlearned about our generation and the world\n01:50 we're building together.\n01:52 But first, the last couple of days have broug

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [36]:
# HuggingFace embeddings run locally - no API key needed
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(chunks, embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [37]:
vector_store.index_to_docstore_id

{0: '9e7e2338-77c6-4705-983b-9e0972c5f35a',
 1: '173570fe-1d15-4e96-925a-783bf0f9ec82',
 2: 'ced248a8-e59a-4ef6-aff8-9fddaf3ff512',
 3: '3b13fa60-f184-40ab-a97d-0c524ee6815d',
 4: '3ae538da-85b0-4102-bdb8-5ef961465fba',
 5: 'ba041649-ce4d-4610-b2b7-b466405e6050',
 6: '84391d08-6508-4c49-8334-910243079464',
 7: '16f94105-e989-49ac-b1ca-f01828cdc14e',
 8: '1a8efcde-a2f9-471a-b755-ee500ab87858',
 9: '0f727685-baee-4585-bb96-775b6579b4a1',
 10: '316771fe-1324-4121-9028-35d1e3b07ff9',
 11: '3dbd3155-2885-45aa-8505-5cd6418b628e',
 12: 'f3a3921b-827b-4103-ad34-0fdeecb80f91',
 13: 'ee4d4b65-0bce-437c-851b-a8fb030acc5b',
 14: '1aa8196b-fd3c-4c1a-8a12-166678f99e08',
 15: 'b7b35859-17e2-4f7d-a402-f24ce9058dc4',
 16: '4bac49b6-54d6-4cf2-8565-1d0c5575dd71',
 17: 'd8fc57bf-a4fd-4f70-8fd3-8af6851e0997',
 18: 'd9076fb7-dad9-4c09-b844-3a9351a2f2e7',
 19: '08570b9e-4485-4935-af65-3d18c7ec5ae7',
 20: '9496bc1d-30dc-4fbf-9945-6c94bec5776b',
 21: '4c4727e8-9f0d-4d91-8712-34187a91b7a7',
 22: 'ce70e0f3-f7e9-

In [38]:
# Get a document by its ID (use any ID from the dict above)
vector_store.get_by_ids(['7d19eeda-2b00-4e4e-8cd4-98188843ce47'])

[Document(id='7d19eeda-2b00-4e4e-8cd4-98188843ce47', metadata={}, page_content='29:58 It says something about our current situation\nthat I can\'t even say his name because I don\'t\n30:03 want to put him at risk.\n30:07 But if a high school senior who doesn\'t know\nwhat the future holds can do his part to move\n30:13 the world forward, then we owe it to the world\nto do our part too.\n30:46 Before you walk out those gates one last time,\nas we sit in front of Memorial Church, I am\n30:55 reminded of a prayer, Mi Shebeirach, that\nI say whenever I face a challenge, that I\n31:05 sing to my daughter thinking about her future\nwhen I tuck her into bed.\n31:09 It goes:\n"May the source of strength, who blessed the\n31:15 ones before us, help us *find the courage*\nto make our lives a blessing."\n31:22 I hope you find the courage to make your life\na blessing.\n31:30 Congratulations, Class of \'17!\n31:32 Good luck out there.')]

## Step 2 - Retrieval

In [39]:
#making retriver using same vector storw, with search type similarity search,  it will provide you 4 similar vectors
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [40]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7af08464cb60>, search_kwargs={'k': 4})

In [42]:
# To ask quey from retriver
retriever.invoke('what was mark zukerberg purpose') #it will return 4 most similar document

# retriver always get query as input and provide document as output

[Document(id='3ae538da-85b0-4102-bdb8-5ef961465fba', metadata={}, page_content='commencement about finding your purpose.\n05:39 We\'re millennials.\n05:40 We\'ll try to do that instinctively.\n05:43 Instead, I\'m here to tell you finding your\npurpose isn\'t enough.\n05:50 The challenge for our generation is creating\na world where everyone has a sense of purpose.\n05:57 One of my favorite stories is when John F\nKennedy visited the NASA space center, he\n06:05 saw a janitor carrying a broom and he walked\nover and asked what he was doing.\n06:08 The janitor responded: "Mr. President, I\'m\nhelping put a man on the moon".\n06:15 Purpose is that sense that we are part of\nsomething bigger than ourselves, that we are\n06:22 needed, that we have something better ahead\nto work for.\n06:26 Purpose is what creates true happiness.\n06:31 You\'re graduating at a time when this is especially\nimportant.\n06:34 When our parents graduated, purpose reliably\ncame from your job, your church, your 

## Step 3 - Augmentation

In [50]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

In [46]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [43]:
question          = "Purpose meaning by mark zukerberg"
retrieved_docs    = retriever.invoke(question) # passing question to retriver it wil give 4 docs

In [ ]:
retrieved_docs

In [44]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs) #just concatinating the 4 docs and making a big string
context_text

'commencement about finding your purpose.\n05:39 We\'re millennials.\n05:40 We\'ll try to do that instinctively.\n05:43 Instead, I\'m here to tell you finding your\npurpose isn\'t enough.\n05:50 The challenge for our generation is creating\na world where everyone has a sense of purpose.\n05:57 One of my favorite stories is when John F\nKennedy visited the NASA space center, he\n06:05 saw a janitor carrying a broom and he walked\nover and asked what he was doing.\n06:08 The janitor responded: "Mr. President, I\'m\nhelping put a man on the moon".\n06:15 Purpose is that sense that we are part of\nsomething bigger than ourselves, that we are\n06:22 needed, that we have something better ahead\nto work for.\n06:26 Purpose is what creates true happiness.\n06:31 You\'re graduating at a time when this is especially\nimportant.\n06:34 When our parents graduated, purpose reliably\ncame from your job, your church, your community.\n06:43 But today, technology and automation are eliminating\nmany jo

In [47]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [48]:
final_prompt

StringPromptValue(text='\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don\'t know.\n\n      commencement about finding your purpose.\n05:39 We\'re millennials.\n05:40 We\'ll try to do that instinctively.\n05:43 Instead, I\'m here to tell you finding your\npurpose isn\'t enough.\n05:50 The challenge for our generation is creating\na world where everyone has a sense of purpose.\n05:57 One of my favorite stories is when John F\nKennedy visited the NASA space center, he\n06:05 saw a janitor carrying a broom and he walked\nover and asked what he was doing.\n06:08 The janitor responded: "Mr. President, I\'m\nhelping put a man on the moon".\n06:15 Purpose is that sense that we are part of\nsomething bigger than ourselves, that we are\n06:22 needed, that we have something better ahead\nto work for.\n06:26 Purpose is what creates true happiness.\n06:31 You\'re graduating at a time when this is es

## Step 4 - Generation

In [51]:
answer = llm.invoke(final_prompt)
print(answer.content)

According to the transcript, Mark Zuckerberg defines purpose as "that sense that we are part of something bigger than ourselves, that we are needed, that we have something better ahead to work for." He also states that "Purpose is what creates true happiness."
